> **Data availability:** The unaligned intermediate datasets this notebook reads (`evaluation_spectra_combined.xlsx`, `raman_consensus_spectra.xlsx`, `raman_valid_samples_combined.xlsx`, `raman_functional_intensity_matrix_bspline_n4.xlsx`) are not included in this repository. Its aligned outputs are the datasets provided in `data/`. This notebook is included to document the processing steps; its file paths refer to the original project layout.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

## 1. Load the Four Final-Folder Datasets

In [ ]:
eval_df = pd.read_excel('evaluation_spectra_combined.xlsx', sheet_name='Spectra Preprocessed', index_col=0)
cons_df = pd.read_excel('Final/raman_consensus_spectra.xlsx', index_col=0)
comb_df = pd.read_excel('raman_valid_samples_combined.xlsx', sheet_name='Spectra Functional', index_col=0)

# Functional intensity matrix is stored sample x wavenumber — transpose to wavenumber x sample
func_wide = pd.read_excel('Final/raman_functional_intensity_matrix_bspline_n4.xlsx', index_col=0).T
func_wide.index = func_wide.index.astype(float)

datasets = {
    'Eval_Preprocessed':                             eval_df,
    'raman_consensus_spectra':                        cons_df,
    'raman_functional_intensity_matrix_bspline_n4':   func_wide,
    'raman_valid_samples_combined':                   comb_df,
}

for name, df in datasets.items():
    print(f'{name:<50} {df.shape[0]:>5} pts x {df.shape[1]:>3} cols   '
          f'range {df.index.min():>9.2f} - {df.index.max():>9.2f} cm-1')

## 2. Determine the Common Grid

In [ ]:
# Common grid: integer cm-1 points covered natively by every dataset (no extrapolation)
common_lo = int(np.ceil(max(df.index.min() for df in datasets.values())))
common_hi = int(np.floor(min(df.index.max() for df in datasets.values())))
common_grid = np.arange(common_lo, common_hi + 1)

print(f'Common grid: {common_lo} - {common_hi} cm-1 ({len(common_grid)} points, 1 cm-1 spacing)')
print('This is the intersection of all four native ranges above — the widest range all')
print('four datasets can be compared over without extrapolating beyond their measured data.')

## 3. Regrid Each Dataset onto the Common Grid

In [ ]:
def regrid(df, grid):
    out = {col: interp1d(df.index.values, df[col].values, kind='linear')(grid) for col in df.columns}
    return pd.DataFrame(out, index=grid)

eval_aligned = regrid(eval_df, common_grid)
cons_aligned = regrid(cons_df, common_grid)
comb_aligned = regrid(comb_df, common_grid)
func_aligned = func_wide.loc[common_grid]   # already on an integer grid — subset only

aligned = {
    'Eval_Preprocessed':                           eval_aligned,
    'raman_consensus_spectra':                      cons_aligned,
    'raman_functional_intensity_matrix_bspline_n4': func_aligned,
    'raman_valid_samples_combined':                 comb_aligned,
}

for name, df in aligned.items():
    print(f'{name:<50} {df.shape}')

assert all((df.index == common_grid).all() for df in aligned.values())
print('\nAll four datasets now share an identical wavenumber index.')

## 4. Sanity Check

Confirm the interpolation didn't distort the spectra — the aligned consensus Arithmetic Mean should still track its pre-alignment curve closely, and the mean spectra of the four datasets should correlate strongly (they all represent the same class of milk / adulterant Raman spectra).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(cons_df.index, cons_df['Arithmetic Mean'], alpha=0.5, linewidth=1.5, color='steelblue',
        label='Consensus (native grid)')
ax.plot(common_grid, cons_aligned['Arithmetic Mean'], linewidth=1.0, color='crimson',
        label='Consensus (aligned to common grid)')
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Alignment Sanity Check — Arithmetic Mean Consensus, Native vs Common Grid')
ax.legend()
plt.tight_layout()
plt.show()

mean_eval = eval_aligned.mean(axis=1)
mean_cons = cons_aligned.mean(axis=1)
mean_func = func_aligned.mean(axis=1)
mean_comb = comb_aligned.mean(axis=1)
corr = pd.DataFrame({
    'Eval': mean_eval,
    'Consensus': mean_cons,
    'Functional': mean_func,
    'Combined': mean_comb,
}).corr()
print('Correlation between dataset mean spectra on the common grid:')
print(corr.round(4))

## 5. Save Common-Grid-Aligned Copies

The originals are left untouched (other notebooks — `raman_qc.ipynb`, `raman_decomposition.ipynb` — depend on their native grids). Aligned copies, sharing the identical integer wavenumber index above, are saved into `Final/` for cross-dataset statistical analysis.

In [ ]:
aligned_paths = {
    'Eval_Preprocessed':                           f'Final/Eval_Preprocessed_aligned_{common_lo}_{common_hi}.xlsx',
    'raman_consensus_spectra':                      f'Final/raman_consensus_spectra_aligned_{common_lo}_{common_hi}.xlsx',
    'raman_functional_intensity_matrix_bspline_n4': f'Final/raman_functional_intensity_matrix_bspline_n4_aligned_{common_lo}_{common_hi}.xlsx',
    'raman_valid_samples_combined':                 f'Final/raman_valid_samples_combined_aligned_{common_lo}_{common_hi}.xlsx',
}

align_readme = pd.DataFrame({'Value': [
    f'Common integer wavenumber grid, {common_lo}-{common_hi} cm^-1 ({len(common_grid)} points).',
    'Linear interpolation (scipy.interpolate.interp1d) from each dataset\'s native grid onto the common grid.',
    'Grid bounds are the intersection of the four native ranges above — no extrapolation.',
    ', '.join(aligned_paths.values()),
]}, index=['Common grid', 'Interpolation', 'Grid selection', 'Sibling aligned files'])
align_readme.index.name = 'Field'

for name, path in aligned_paths.items():
    with pd.ExcelWriter(path) as writer:
        aligned[name].to_excel(writer, sheet_name='Aligned Spectra')
        align_readme.to_excel(writer, sheet_name='README')
    print(f'Saved -> {path}  ({aligned[name].shape[0]} points x {aligned[name].shape[1]} columns)')

## 6. Recalibrate the Held-Out Evaluation Set (per-wavenumber batch standardisation)

The evaluation panel was acquired on a different setup (100 % power / 3 s / 96-well plate) and carries a
**shared per-wavenumber batch offset** relative to the training milk — every eval spectrum (authentic and
adulterated alike) is displaced from the training consensus by essentially the same vector, which destroys
class separation for any distance-to-consensus classifier.

This step removes that offset with a **label-free per-wavenumber batch standardisation**: for each
wavenumber, the eval batch is standardised to zero-mean/unit-variance using the eval batch's own
per-wavenumber statistics, then re-scaled to the **authentic training** population's per-wavenumber mean
and spread,

$$s'_{\text{eval}}(w)=\frac{s_{\text{eval}}(w)-\bar e(w)}{\sigma_e(w)}\,\sigma_v(w)+\bar v(w).$$

It is **self-contained**: it reads the current canonical aligned files (the up-to-date datasets consumed
downstream) rather than the interpolated frames above, and writes a sibling
`Eval_Preprocessed_aligned_402_3499_recalibrated.xlsx`. The raw aligned eval file is preserved. The
recalibrated file is what `raman_loo_authenticity.ipynb` and `adulterated_spectra_classification.ipynb`
consume.

In [ ]:
import numpy as np
import pandas as pd

ALIGNED_DIR = 'Final/Aligned Final Datasets'
GRID = '402_3499'

# Canonical, up-to-date aligned datasets used downstream (authentic training + held-out eval)
_valid = pd.read_excel(f'{ALIGNED_DIR}/raman_valid_samples_combined_aligned_{GRID}.xlsx',
                       sheet_name='Aligned Spectra', index_col=0)
_eval  = pd.read_excel(f'{ALIGNED_DIR}/Eval_Preprocessed_aligned_{GRID}.xlsx',
                       sheet_name='Aligned Spectra', index_col=0)
assert (_valid.index == _eval.index).all(), 'valid/eval grids must match'

# Per-wavenumber statistics
v_mean = _valid.mean(axis=1)
v_std  = _valid.std(axis=1, ddof=1).clip(lower=1e-9)
e_mean = _eval.mean(axis=1)
e_std  = _eval.std(axis=1, ddof=1).clip(lower=1e-9)

# Per-wavenumber batch standardisation: eval batch -> authentic training scale
eval_recal = _eval.sub(e_mean, axis=0).div(e_std, axis=0).mul(v_std, axis=0).add(v_mean, axis=0)

before = np.linalg.norm(_eval.mean(axis=1).values - v_mean.values)
after  = np.linalg.norm(eval_recal.mean(axis=1).values - v_mean.values)
print(f'Authentic training samples : {_valid.shape[1]}')
print(f'Evaluation samples         : {_eval.shape[1]}')
print(f'||eval mean - authentic mean||  before recal : {before:.4f}')
print(f'||eval mean - authentic mean||  after  recal : {after:.4f}')

recal_readme = pd.DataFrame({'Value': [
    'Per-wavenumber batch standardisation of the aligned evaluation set.',
    "s' = (s - eval_mean_w)/eval_std_w * valid_std_w + valid_mean_w, per wavenumber w.",
    'Target scale: authentic training population (raman_valid_samples_combined_aligned).',
    'Label-free; removes the shared cross-setup per-wavenumber batch offset.',
    f'{ALIGNED_DIR}/Eval_Preprocessed_aligned_{GRID}.xlsx (raw aligned source, preserved).',
]}, index=['Purpose', 'Transform', 'Target', 'Note', 'Source'])
recal_readme.index.name = 'Field'

out_path = f'{ALIGNED_DIR}/Eval_Preprocessed_aligned_{GRID}_recalibrated.xlsx'
with pd.ExcelWriter(out_path) as writer:
    eval_recal.to_excel(writer, sheet_name='Aligned Spectra')
    recal_readme.to_excel(writer, sheet_name='README')
print(f'Saved -> {out_path}  ({eval_recal.shape[0]} points x {eval_recal.shape[1]} columns)')